In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

points = pd.read_parquet(PROCESSED / "points_raw.parquet")
print("Loaded points:", points.shape)

Loaded points: (1415633, 15)


In [2]:
# Shot type codes. Forehand and backhand variants are kept separate
# because tactical behaviour differs between wings.
SHOT_TYPES = {
    'f': 'forehand',            'b': 'backhand',
    'r': 'forehand_slice',      's': 'backhand_slice',
    'v': 'forehand_volley',     'z': 'backhand_volley',
    'o': 'overhead',            'p': 'backhand_overhead',
    'u': 'forehand_dropshot',   'y': 'backhand_dropshot',
    'l': 'forehand_lob',        'm': 'backhand_lob',
    'h': 'forehand_halfvolley', 'i': 'backhand_halfvolley',
    'j': 'forehand_swingvolley','k': 'backhand_swingvolley',
    't': 'trick',               'q': 'unknown',
}

# Error type, recorded on the shot that ended the point (Target 9).
ERROR_TYPES = {'n': 'net', 'w': 'wide', 'd': 'long', 'x': 'wide_and_long'}

# Serve placement. These codes are absolute, not court-side dependent,
# so no separate deuce/ad mapping is required.
SERVE_DIR = {'4': 'Wide', '5': 'Body', '6': 'T'}

# A valid point string must end in one of these.
TERMINATORS = {'*': 'winner', '@': 'unforced_error', '#': 'forced_error'}

In [3]:
# One shot = type letter, optional position markers, optional direction
# digit (1-3), optional depth digit (7-9), optional error code. Markers are
# matched both before and after the digits because charters are not
# consistent about their placement.
SHOT_RE = re.compile(r'([fbrsvzopuylmhijktq])([\+\-\=\^;]*)([123]?)([789]?)([\+\-\=\^;]*)([nwdx]?)')


def parse_point(raw):
    """
    Parse one MCP notation string into serve placement, outcome and shot list.
    Returns None for strings we cannot trust (no valid terminator, no serve).
    """
    if not isinstance(raw, str) or len(raw) == 0:
        return None

    s = raw.strip()

    # A point that does not end in * @ or # was not charted to completion.
    if len(s) == 0 or s[-1] not in TERMINATORS:
        return None

    outcome = TERMINATORS[s[-1]]
    body = s[:-1]

    # Leading 'c' marks a let, which is replayed and carries no information.
    body = re.sub(r'^c+', '', body)

    # Serve direction is the first 4/5/6 digit. 0 means the charter did not
    # record placement, so we keep the point but leave direction blank.
    m = re.match(r'^[SRQPg]*([0456])', body)
    if not m:
        return None
    serve_dir = SERVE_DIR.get(m.group(1))

    shots = []
    for mt in SHOT_RE.finditer(body[m.end():]):
        shots.append({
            'shot_type': SHOT_TYPES.get(mt.group(1), 'other'),
            'direction': mt.group(3) or None,
            'depth': mt.group(4) or None,
            'approach': '+' in (mt.group(2) + mt.group(5)),
            'error_type': ERROR_TYPES.get(mt.group(6)),
        })

    # Rally length counts the serve plus every subsequent shot.
    return {'serve_dir': serve_dir, 'outcome': outcome,
            'shots': shots, 'rally_len': 1 + len(shots)}

In [4]:
# The point actually played is the second serve when one exists, otherwise
# the first. The unused first-serve string only records a fault.
sequence = points['2nd'].fillna(points['1st']).astype(str)
is_second_serve = points['2nd'].notna()

point_rows = []
shot_rows = []

for raw, second, mid, pt, svr, wnr, gen, score in zip(
        sequence, is_second_serve, points['match_id'], points['Pt'],
        points['Svr'], points['PtWinner'], points['gender'], points['Pts']):

    parsed = parse_point(raw)
    if parsed is None:
        continue

    point_rows.append((mid, pt, gen, svr, wnr, second, score,
                       parsed['serve_dir'], parsed['outcome'], parsed['rally_len']))

    # Direction codes name a side of the court, not a direction relative to
    # the hitter. Cross-court vs down-the-line therefore depends on where the
    # previous shot landed: same code = cross-court, different = down-the-line.
    previous_direction = None

    for shot_num, shot in enumerate(parsed['shots'], start=2):
        d = shot['direction']

        if d is None or previous_direction is None:
            pattern = None
        elif d == '2' or previous_direction == '2':
            pattern = 'middle'
        elif d == previous_direction:
            pattern = 'cross_court'
        else:
            pattern = 'down_the_line'

        # Odd-numbered shots are hit by the server, even ones by the returner.
        hitter = svr if shot_num % 2 == 1 else 3 - svr

        shot_rows.append((mid, pt, gen, shot_num, hitter, shot['shot_type'],
                          d, shot['depth'], shot['approach'],
                          shot['error_type'], pattern))

        previous_direction = d

points_parsed = pd.DataFrame(point_rows, columns=[
    'match_id', 'Pt', 'gender', 'server', 'point_winner', 'is_second_serve',
    'Pts', 'serve_direction', 'point_outcome', 'rally_length'])

shots_parsed = pd.DataFrame(shot_rows, columns=[
    'match_id', 'Pt', 'gender', 'shot_number', 'hitter', 'shot_type',
    'direction', 'depth', 'is_approach', 'error_type', 'direction_pattern'])

print("Points parsed:", points_parsed.shape)
print("Shots parsed: ", shots_parsed.shape)

Points parsed: (1353711, 10)
Shots parsed:  (5308401, 11)


In [5]:
# Report coverage explicitly. Points are dropped when the notation lacks a
# terminator or a serve code, meaning the charter did not complete them.
n_total = len(points)
n_kept = len(points_parsed)

print(f"Total points:   {n_total:,}")
print(f"Parsed:         {n_kept:,}  ({100 * n_kept / n_total:.1f}%)")
print(f"Dropped:        {n_total - n_kept:,}  ({100 * (n_total - n_kept) / n_total:.1f}%)")

Total points:   1,415,633
Parsed:         1,353,711  (95.6%)
Dropped:        61,922  (4.4%)


In [6]:
# The parser never sees PtWinner, so it can serve as ground truth.
# If a point ended in an error, the last shot was hit by the loser; if it
# ended in a winner, by the winner. Reconstruct and compare.
last_hitter = np.where(points_parsed['rally_length'] % 2 == 1,
                       points_parsed['server'],
                       3 - points_parsed['server'])

implied_winner = np.where(points_parsed['point_outcome'] == 'winner',
                          last_hitter,
                          3 - last_hitter)

agreement = (implied_winner == points_parsed['point_winner']).mean()
print(f"Parser-implied winner matches PtWinner: {100 * agreement:.2f}%")

Parser-implied winner matches PtWinner: 97.71%


In [7]:
points_parsed.to_parquet(PROCESSED / "points_parsed.parquet", index=False)
shots_parsed.to_parquet(PROCESSED / "shots_parsed.parquet", index=False)
print("Saved both tables to:", PROCESSED)

Saved both tables to: C:\Users\dovyd\Documents\tennis-shot-quality\data\processed
